# PK/PD simulation in Python, all graphs

Teaching example parameters only, not a real drug, not a real patient. This notebook reproduces every graph from Stages 1 to 5 and saves each graph as a PNG file, so the graphs can be uploaded to GitHub.


In [ ]:
!pip install numpy scipy matplotlib pandas -q
print('Install step done.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Teaching parameters only
dose_mg, V_L, CL_L_per_h, ka_per_h, F = 100.0, 50.0, 5.0, 1.0, 1.0
k_per_h = CL_L_per_h / V_L
C0 = dose_mg / V_L
print('k', k_per_h, 'C0', C0, 'half-life', round(float(np.log(2)/k_per_h),2), 'AUC', dose_mg/CL_L_per_h)
# Stage 1 graph
t = np.linspace(0, 48, 200)
C_iv = C0 * np.exp(-k_per_h * t)
plt.figure(); plt.plot(t, C_iv); plt.xlabel('Time (hours)'); plt.ylabel('Concentration (mg per L)'); plt.title('Stage 1: One compartment IV bolus, teaching example'); plt.grid(True); plt.savefig('pkpd_stage1_iv_bolus.png', dpi=150); plt.show()
print('Saved pkpd_stage1_iv_bolus.png')


In [ ]:
# Stage 2 graph, oral, Bateman function
t = np.linspace(0, 48, 400)
C_oral = (F * dose_mg * ka_per_h / (V_L * (ka_per_h - k_per_h))) * (np.exp(-k_per_h * t) - np.exp(-ka_per_h * t))
plt.figure(); plt.plot(t, C_oral); plt.xlabel('Time (hours)'); plt.ylabel('Concentration (mg per L)'); plt.title('Stage 2: Oral dosing, teaching example'); plt.grid(True); plt.savefig('pkpd_stage2_oral.png', dpi=150); plt.show()
print('Cmax', round(float(C_oral.max()),2), 'Tmax', round(float(t[C_oral.argmax()]),2))
print('Saved pkpd_stage2_oral.png')


In [ ]:
# Stage 3 graph, multiple dosing, 100 mg every 12 hours, 10 doses, superposition
t = np.linspace(0, 120, 1200)
C_multi = np.zeros_like(t)
for d in range(10):
    td = t - d*12
    td = np.where(td>=0, td, 0)
    # oral curve for each dose, zero before its dose time
    c = (F * dose_mg * ka_per_h / (V_L * (ka_per_h - k_per_h))) * (np.exp(-k_per_h * td) - np.exp(-ka_per_h * td))
    c = np.where(t>=d*12, c, 0)
    C_multi += c
plt.figure(); plt.plot(t, C_multi); plt.xlabel('Time (hours)'); plt.ylabel('Concentration (mg per L)'); plt.title('Stage 3: Multiple dosing to steady state, teaching example'); plt.grid(True); plt.savefig('pkpd_stage3_multiple_dosing.png', dpi=150); plt.show()
print('Saved pkpd_stage3_multiple_dosing.png')


In [ ]:
# Stage 4 graphs, Emax
Emax, EC50 = 100.0, 2.0
def emax_effect(C): return Emax * C / (EC50 + C)
C_range = np.linspace(0, 20, 200)
plt.figure(); plt.plot(C_range, emax_effect(C_range)); plt.xlabel('Concentration (mg per L)'); plt.ylabel('Effect (teaching units)'); plt.title('Stage 4: Emax concentration effect curve, teaching example'); plt.grid(True); plt.savefig('pkpd_stage4_emax_curve.png', dpi=150); plt.show()
print('Effect at EC50', emax_effect(EC50), 'Effect at 10x EC50', round(float(emax_effect(10*EC50)),1))
t = np.linspace(0, 48, 400)
C = (F * dose_mg * ka_per_h / (V_L * (ka_per_h - k_per_h))) * (np.exp(-k_per_h * t) - np.exp(-ka_per_h * t))
E = emax_effect(C)
plt.figure(); plt.plot(t, C, label='Concentration'); plt.plot(t, E, label='Effect'); plt.xlabel('Time (hours)'); plt.legend(); plt.title('Stage 4: Concentration and effect over time, teaching example'); plt.grid(True); plt.savefig('pkpd_stage4_effect_time.png', dpi=150); plt.show()
print('Peak effect', round(float(E.max()),1))
print('Saved pkpd_stage4_emax_curve.png and pkpd_stage4_effect_time.png')


In [ ]:
# Stage 5 graphs, sensitivity
def oral_curve(tt, dose_mg=100.0, V_L=50.0, CL_L_per_h=5.0, ka_per_h=1.0, F=1.0):
    kk = CL_L_per_h / V_L
    return (F * dose_mg * ka_per_h / (V_L * (ka_per_h - kk))) * (np.exp(-kk * tt) - np.exp(-ka_per_h * tt))
t = np.linspace(0, 48, 400)
plt.figure(); plt.plot(t, oral_curve(t), label='Base case'); plt.plot(t, oral_curve(t, CL_L_per_h=10.0), label='Clearance doubled'); plt.plot(t, oral_curve(t, V_L=100.0), label='Volume doubled'); plt.plot(t, oral_curve(t, ka_per_h=0.3), label='ka slower (0.3 per hour)'); plt.xlabel('Time (hours)'); plt.ylabel('Concentration (mg per L)'); plt.title('Stage 5: Sensitivity, one parameter changed at a time'); plt.legend(); plt.grid(True); plt.savefig('pkpd_stage5_sensitivity.png', dpi=150); plt.show()
C_range = np.linspace(0, 20, 200)
plt.figure()
for ec50 in [1.0, 2.0, 5.0]:
    plt.plot(C_range, 100.0 * C_range / (ec50 + C_range), label='EC50 = ' + str(ec50))
plt.xlabel('Concentration (mg per L)'); plt.ylabel('Effect (teaching units)'); plt.title('Stage 5: EC50 sensitivity, Emax model'); plt.legend(); plt.grid(True); plt.savefig('pkpd_stage5_ec50_sensitivity.png', dpi=150); plt.show()
print('Saved pkpd_stage5_sensitivity.png and pkpd_stage5_ec50_sensitivity.png')


## MY RESULTS, all graphs

All 7 graphs are saved as PNG files in the Colab Files panel on the left. Download all 7 PNG files, then upload them to GitHub with the README. Do not use these teaching graphs to dose a real patient.